#### TOOLS

Models can request to call tools that performs tasks as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions(often a json schema)
2. a function or coroutine to execute

In [3]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model(
     "groq:llama-3.1-8b-instant",
    
    
)

response=model.invoke("what is capital of india")
response.content
# we have to integrate a tool into this 




'The capital of India is New Delhi.'

### 🛠️ LangChain Tool Schema Guide

Here is the structured guide on how to create and understand tools in LangChain.

---

#### 1. Code Template

Use the `@tool` decorator to convert a standard Python function into an LLM-accessible tool.

```python
from langchain.tools import tool

@tool
def function_name(parameter1: type, parameter2: type, ...) -> return_type:
    """
    Description of what the tool does.
    This docstring is extremely important.
    """
    # Your logic
    return result
```

---

#### 2. Component Breakdown

| Part | Purpose |
| :--- | :--- |
| **`@tool`** | Converts a normal Python function into a LangChain tool. |
| **`function_name`** | The name the LLM sees and may choose to call. |
| **Parameters** | Inputs the LLM should provide. Type hints (`str`, `int`, etc.) are critical. |
| **Return type** (`-> str`) | Specifies what data type the tool returns. |
| **Docstring** | **Most important part.** It tells the LLM exactly when and why to use the tool. |
| **Function body** | The actual executable Python code implementation. |

---

#### 3. Internal LLM Representation

LangChain automatically extracts your Python metadata and parses it into a standard JSON schema that the LLM reads internally:

```json
{
  "name": "get_weather",
  "description": "Get the current weather for a given location.",
  "parameters": {
    "type": "object",
    "properties": {
      "location": {
        "type": "string"
      }
    },
    "required": ["location"]
  }
}
```


In [ ]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}"

llm = model.bind_tools([get_weather])

response = llm.invoke("What's the weather in Mumbai?")

print(response)
print(response.tool_calls)

content='' additional_kwargs={'tool_calls': [{'id': 'jxtddq19b', 'function': {'arguments': '{"location":"Mumbai"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 219, 'total_tokens': 234, 'completion_time': 0.037899884, 'completion_tokens_details': None, 'prompt_time': 0.011084646, 'prompt_tokens_details': None, 'queue_time': 0.161648263, 'total_time': 0.04898453}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f863e-a9ea-78b3-aee6-fa8f12484d87-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': 'jxtddq19b', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 219, 'output_tokens': 15, 'total_tokens': 234}
[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': 'jxtddq19b', 'type': 'tool_call'}]


### 🔄 Methods to Bind LLMs and Tools

When working with LangChain, there are two primary methods used to connect your language model to custom tools.

---

#### 1. `bind_tools()` (Manual Flow)
With `bind_tools()`, you attach the tools to the model schema. However, you must manage the execution loop, look for tool calls, and run them manually.

```python
# Bind the tools to your chosen chat model
model = init_chat_model("groq:llama-3.3-70b-versatile")
llm = model.bind_tools([get_weather, tool2, tool3])

# Invoke the model
response = llm.invoke("What's the weather in Mumbai?")

# Manually inspect the requested tool calls
print(response.tool_calls)
```

---

#### 2. `create_agent()` (Automatic Loop)
The `create_agent()` method abstracts the entire tool-calling cycle. It sends the prompt, reads the tool call request, executes the underlying Python function, and pipes the result back to the model automatically.

```python
from langchain.agents import create_agent

# Initialize the automated agent with its tools
agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[get_weather]
)

# Invoke the agent directly
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user", 
                "content": "What's the weather in Mumbai?"
            }
        ]
    }
)

print(response)
```

> 💡 **Key Takeaway**: By using `create_agent()`, you do not have to manually parse `tool_calls` or manually run your tool logic.


In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool


agent = create_agent(
    model=model, # we initialized the model=init_chat_model in the first cell itself
    tools=[get_weather]
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in Mumbai?"}]}
)

print(response)

{'messages': [HumanMessage(content="What's the weather in Mumbai?", additional_kwargs={}, response_metadata={}, id='e3e78885-90a2-497e-967d-f57a2d9ec467'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '1dnatc3zt', 'function': {'arguments': '{"location":"Mumbai"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 219, 'total_tokens': 234, 'completion_time': 0.036710993, 'completion_tokens_details': None, 'prompt_time': 0.010596098, 'prompt_tokens_details': None, 'queue_time': 0.161295669, 'total_time': 0.047307091}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f8640-1c1e-7592-9f8a-b22dbe869d27-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': '1dnatc3zt', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata=

AIMessage(
    content='',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'location': 'Mumbai'}
        }
    ]
)

In [ ]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What's the weather in Mumbai?"
            }
        ]
    }
)

messages = response["messages"]

print("=" * 60)
print("1. USER MESSAGE")
print("=" * 60)
print(messages[0].content)

print("\n" + "=" * 60)
print("2. LLM DECIDES TO CALL TOOL")
print("=" * 60)
print(messages[1].tool_calls)

print("\n" + "=" * 60)
print("3. TOOL RETURNS")
print("=" * 60)
print(messages[2].content)

print("\n" + "=" * 60)
print("4. FINAL LLM RESPONSE")
print("=" * 60)
print(messages[3].content)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01ky290y2sey39gy8yfwcqhjb6` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99542, Requested 670. Please try again in 3m3.168s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

#### An AI Agent makes 2 calls to the LLM, hence mre tokens and rate limit gets reached faster. 
1. When the user asks a question, llm first processes it and then it sends to whichever tool it needs
2. when the tool gets back with its answer to the llm, this is the second call made then llm prepares the answer and sends it back to us

![Basic Agent](Images/basic_agent.png)